In [24]:
import sys, os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pylab as plt
from scipy.ndimage.measurements import center_of_mass

repo_path = os.path.abspath('~/ukbb-pulmonary-artery/DeepCMR')
assert os.path.isdir(repo_path)
if not repo_path in sys.path: sys.path.append(repo_path)

from utils import visualizer 

%config Completer.use_jedi = False

Let's find all subjects ready for training and testing: 

In [ ]:
data_path = '{deepcmr_data_root}'

PatientNames = np.unique([os.path.basename(name).strip('.nii.gz').strip('_gt') 
                          for name in glob.glob(os.path.join(data_path, 'OrthancDicomStorageNiftis', '*'))])

PatientNames_validation = np.unique([os.path.basename(name).strip('.nii.gz').strip('_gt') 
                                     for name in glob.glob(os.path.join(data_path, 'OrthancDicomStorage-PracticeNiftis', '*'))])

print(len(PatientNames), 'training subjects.', len(PatientNames_validation), 'testing subjects.')

### Training

Let $V$ denote the MR image, and let $M$ denote the mask of the pulmonary artery (i.e., the segmentation): 

In [ ]:
for patient_id, PatientName in enumerate(PatientNames):
    V_nifti = nib.load(os.path.join(data_path, 'OrthancDicomStorageNiftis', PatientName + '.nii.gz'))
    M_nifti = nib.load(os.path.join(data_path, 'OrthancDicomStorageNiftis', PatientName + '_gt.nii.gz'))

    V = V_nifti.get_fdata()
    M = M_nifti.get_fdata()
    
    
    cx, cy = center_of_mass(M)[:2]
    v = V[int(cx)-20:int(cx)+20,int(cy)-20:int(cy)+20].squeeze()
    m = M[int(cx)-20:int(cx)+20,int(cy)-20:int(cy)+20].squeeze()

    
    fig, ax = plt.subplots(1,10,figsize=(10*5,5))
    for i, t in enumerate(range(1,50,5)):
        ax[i].imshow(v[:,:,t], cmap='gray')
        visualizer.PlotContours(ax[i], m[:,:,t])
        ax[i].axis('off')
        if i == 0:
            ax[i].set_title(PatientName, fontsize=40)
        else:
            ax[i].set_title(patient_id, fontsize=40)
    plt.show()




    